In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Hello world

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na requirements.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Ang halimbawang ito ay may dalawang bahagi. Una, gagawa ka ng simpleng quantum program at patakbuhin ito sa isang quantum processing unit (QPU). Dahil ang tunay na quantum research ay nangangailangan ng mas matibay na mga programa, sa ikalawang bahagi ([I-scale sa malaking bilang ng qubits](#scale-to-large-numbers-of-qubits)), iscale mo ang simpleng programa hanggang sa utility level.

## I-install at mag-authenticate
1. Kung hindi mo pa nai-install ang Qiskit, hanapin ang mga instruksyon sa gabay na [Quickstart](/guides/quick-start).

    - I-install ang Qiskit Runtime para makapagpatakbo ng mga job sa quantum hardware:

        ```bash
        pip install qiskit-ibm-runtime
        ```

    - Mag-set up ng environment para makapagpatakbo ng mga Jupyter notebook nang lokal:

        ```bash
        pip install jupyter
        ```

2. I-set up ang iyong authentication para ma-access ang quantum hardware sa pamamagitan ng libreng [Open Plan](/guides/plans-overview#open-plan).

    (Kung nakatanggap ka ng email na imbitasyon para sumali sa isang account, sundin ang [mga hakbang para sa mga inimbitahang user](/guides/cloud-setup-invited) sa halip.)

    - Pumunta sa [IBM Quantum Platform](https://quantum.cloud.ibm.com/) para mag-log in o lumikha ng account.
         > **Note:** Kung kumokonekta ka sa pamamagitan ng proxy server, kailangan mong gumamit ng Qiskit Runtime v0.44.0 o mas bago.
    - I-generate ang iyong API key (tinatawag din itong *API token*) sa [dashboard](https://quantum.cloud.ibm.com/), tapos kopyahin ito sa isang ligtas na lugar.
    - Pumunta sa pahina ng [Instances](https://quantum.cloud.ibm.com/instances) at hanapin ang instance na gusto mong gamitin. I-hover ang CRN nito at i-click para kopyahin.

    - I-save ang iyong mga kredensyal nang lokal gamit ang code na ito:

        ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        QiskitRuntimeService.save_account(
        token="<your-api-key>", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
        instance="<CRN>", # Optional
        )
        ```

3. Maaari mo na ngayong gamitin ang Python code na ito sa tuwing gusto mong mag-authenticate sa Qiskit Runtime Service:
    ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        # Run every time you need the service
        service = QiskitRuntimeService()
    ```
> **Info:** Kung gumagamit ka ng pampublikong computer o iba pang hindi ligtas na environment, sundin ang [mga instruksyon para sa manual na authentication](/guides/cloud-setup-untrusted) para mapanatiling ligtas ang iyong mga kredensyal.
## Gumawa at magpatakbo ng simpleng quantum program
Ang apat na hakbang sa pagsusulat ng quantum program gamit ang mga Qiskit pattern ay:

1.  I-map ang problema sa quantum-native na format.

2.  I-optimize ang mga Circuit at operator.

3.  Mag-execute gamit ang quantum primitive function.

4.  Suriin ang mga resulta.

### Hakbang 1. I-map ang problema sa quantum-native na format
Sa isang quantum program, ang *quantum circuits* ang katutubong format para kumatawan sa mga quantum na instruksyon, at ang *mga operator* naman ang kumakatawan sa mga observable na susukatín. Kapag gumagawa ng Circuit, karaniwang gagawa ka ng bagong [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) na object, tapos magdadagdag ng mga instruksyon dito nang sunud-sunod.
Ang sumusunod na code cell ay gumagawa ng Circuit na nagpo-produce ng *Bell state*, na isang state kung saan dalawang qubit ang ganap na naka-entangle sa isa't isa.

> **Note:** Ang Qiskit SDK ay gumagamit ng LSb 0 bit numbering kung saan ang $n^{th}$ na digit ay may value na $1 \ll n$ o $2^n$. Para sa karagdagang detalye, tingnan ang paksa na [Bit-ordering sa Qiskit SDK](/guides/bit-ordering).

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from matplotlib import pyplot as plt
# Uncomment the next line if you want to use a simulator:
# from qiskit_ibm_runtime.fake_provider import FakeBelemV2


# Create a new circuit with two qubits
qc = QuantumCircuit(2)

# Add a Hadamard gate to qubit 0
qc.h(0)

# Perform a controlled-X gate on qubit 1, controlled by qubit 0
qc.cx(0, 1)

# Return a drawing of the circuit using MatPlotLib ("mpl").
# These guides are written by using Jupyter notebooks, which
# display the output of the last line of each cell.
# If you're running this in a script, use `print(qc.draw())` to
# print a text drawing.
qc.draw("mpl")

<Image src="../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg)

Tingnan ang [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) sa dokumentasyon para sa lahat ng available na operasyon.
Kapag gumagawa ng mga quantum circuit, kailangan mo ring isaalang-alang kung anong uri ng data ang gusto mong ibalik pagkatapos ng execution. Nagbibigay ang Qiskit ng dalawang paraan para ibalik ang data: maaari kang makakuha ng probability distribution para sa isang set ng mga qubit na pipiliin mong sukatin, o maaari kang makakuha ng expectation value ng isang observable. Ihanda ang iyong workload para sukatin ang iyong Circuit sa isa sa dalawang paraang ito gamit ang [Qiskit primitives](/guides/get-started-with-primitives) (ipinapaliwanag nang detalye sa [Hakbang 3](#step-3-execute-using-the-quantum-primitives)).

Ang halimbawang ito ay sumusukat ng mga expectation value gamit ang `qiskit.quantum_info` submodule, na tinutukoy gamit ang mga operator (mga mathematical na object na ginagamit para kumatawan sa isang aksyon o proseso na nagbabago ng quantum state). Ang sumusunod na code cell ay gumagawa ng anim na two-qubit na Pauli operator: `IZ`, `IX`, `ZI`, `XI`, `ZZ`, at `XX`.

In [2]:
# Set up six different observables.

observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

> **Note:** Dito, ang isang bagay tulad ng `ZZ` operator ay isang shorthand para sa tensor product na $Z\otimes Z$, na nangangahulugang sinusukat ang Z sa qubit 1 at Z sa qubit 0 nang sabay, at nakakakuha ng impormasyon tungkol sa ugnayan ng qubit 1 at qubit 0. Ang mga expectation value na tulad nito ay karaniwang isinusulat din bilang $\langle Z_1 Z_0 \rangle$.
> 
> Kung ang state ay naka-entangle, ang sukat ng $\langle Z_1 Z_0 \rangle$ ay dapat na iba sa sukat ng $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$. Para sa partikular na entangled state na ginawa ng ating Circuit na inilarawan sa itaas, ang sukat ng $\langle Z_1 Z_0 \rangle$ ay dapat na 1 at ang sukat ng $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$ ay dapat na zero.
<span id="optimize"></span>

### Hakbang 2. I-optimize ang mga Circuit at operator

Kapag nag-e-execute ng mga Circuit sa isang device, mahalaga na i-optimize ang set ng mga instruksyon na nilalaman ng Circuit at bawasan ang kabuuang depth (humigit-kumulang na bilang ng mga instruksyon) ng Circuit. Tinitiyak nito na makukuha mo ang pinakamahusay na mga resulta sa pamamagitan ng pagbabawas ng mga epekto ng error at noise. Bukod dito, ang mga instruksyon ng Circuit ay dapat na sumunod sa [Instruction Set Architecture (ISA)](/guides/transpile#instruction-set-architecture) ng backend device at dapat isaalang-alang ang mga basis gate at qubit connectivity ng device.

Ang sumusunod na code ay nagpapainstansya ng tunay na device para makapag-submit ng job at nagbabago ng Circuit at mga observable para tumugma sa ISA ng backend na iyon. Kinakailangan na [na-save mo na ang iyong mga kredensyal](/guides/cloud-setup) bago ito.

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(simulator=False, operational=True)

# Convert to an ISA circuit and layout-mapped observables.
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

isa_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg)

### Hakbang 3. Mag-execute gamit ang mga quantum primitive
Ang mga quantum computer ay maaaring mag-produce ng mga random na resulta, kaya karaniwang nangongolekta ka ng sample ng mga output sa pamamagitan ng paulit-ulit na pagpapatakbo ng Circuit. Maaari mong i-estimate ang value ng observable gamit ang `Estimator` class. Ang `Estimator` ay isa sa dalawang [primitive](/guides/get-started-with-primitives); ang isa ay ang `Sampler`, na maaaring gamitin para makakuha ng data mula sa isang quantum computer. Ang mga object na ito ay may `run()` na method na nagpapatupad ng piniling mga Circuit, observable, at parameter (kung applicable), gamit ang isang [primitive unified bloc (PUB).](/guides/primitives#sampler)

In [4]:
# Construct the Estimator instance.

estimator = Estimator(mode=backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 5000

mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

# One pub, with one circuit to run against five different observables.
job = estimator.run([(isa_circuit, mapped_observables)])

# Use the job ID to retrieve your job data later
print(f">>> Job ID: {job.job_id()}")

>>> Job ID: d5k96q4jt3vs73ds5tgg


After a job is submitted, you can wait until either the job is completed within your current python instance, or use the `job_id` to retrieve the data at a later time.  (See the [section on retrieving jobs](/docs/guides/monitor-job#retrieve-job-results-at-a-later-time) for details.)

After the job completes, examine its output through the job's `result()` attribute.

In [5]:
# This is the result of the entire submission.  You submitted one Pub,
# so this contains one inner result (and some metadata of its own).
job_result = job.result()

# This is the result from our single pub, which had six observables,
# so contains information on all six.
pub_result = job.result()[0]

In [6]:
# Check there are six observables.
# If not, edit the comments in the previous cell and update this test.
assert len(pub_result.data.evs) == 6

<Admonition type="note" title="Alternative: run the example using a simulator">
  When you run your quantum program on a real device, your workload must wait in a queue before it runs. To save time, you can instead use the following code to run this small workload on the [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) with the Qiskit Runtime local testing mode. Note that this is only possible for a small circuit. When you scale up in the next section, you will need to use a real device.

  ```python

  # Use the following code instead if you want to run on a simulator:

  from qiskit_ibm_runtime.fake_provider import FakeBelemV2
  backend = FakeBelemV2()
  estimator = Estimator(backend)

  # Convert to an ISA circuit and layout-mapped observables.

  pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
  isa_circuit = pm.run(qc)
  mapped_observables = [
      observable.apply_layout(isa_circuit.layout) for observable in observables
  ]

  job = estimator.run([(isa_circuit, mapped_observables)])
  result = job.result()

  # This is the result of the entire submission.  You submitted one Pub,
  # so this contains one inner result (and some metadata of its own).

  job_result = job.result()

  # This is the result from our single pub, which had five observables,
  # so contains information on all five.

  pub_result = job.result()[0]
  ```
</Admonition>

### Step 4. Analyze the results

The analyze step is typically where you might post-process your results using, for example, measurement error mitigation or zero noise extrapolation (ZNE). You might feed these results into another workflow for further analysis or prepare a plot of the key values and data. In general, this step is specific to your problem.  For this example, plot each of the expectation values that were measured for our circuit.

The expectation values and standard deviations for the observables you specified to Estimator are accessed through the job result's `PubResult.data.evs` and `PubResult.data.stds` attributes. To obtain the results from Sampler, use the `PubResult.data.meas.get_counts()` function, which will return a `dict` of measurements in the form of bitstrings as keys and counts as their corresponding values. For more information, see [Get started with Sampler.](/docs/guides/get-started-with-primitives#get-started-with-sampler)

In [ ]:
# Plot the result

values = pub_result.data.evs

errors = pub_result.data.stds

# plotting graph
plt.plot(observables_labels, values, "-o")
plt.xlabel("Observables")
plt.ylabel("Values")
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg" alt="Output of the previous code cell" />

> **Note:** Kapag pinatakbo mo ang iyong quantum program sa isang tunay na device, ang iyong workload ay dapat munang maghintay sa pila bago tumakbo. Para makatipid ng oras, maaari mo ring gamitin ang sumusunod na code para patakbuhin ang maliit na workload na ito sa [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) gamit ang Qiskit Runtime local testing mode. Tandaan na posible lang ito para sa maliit na Circuit. Kapag nag-scale up ka sa susunod na seksyon, kailangan mong gumamit ng tunay na device.
> 
>   ```python
> 
>   # Use the following code instead if you want to run on a simulator:
> 
>   from qiskit_ibm_runtime.fake_provider import FakeBelemV2
>   backend = FakeBelemV2()
>   estimator = Estimator(backend)
> 
>   # Convert to an ISA circuit and layout-mapped observables.
> 
>   pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
>   isa_circuit = pm.run(qc)
>   mapped_observables = [
>       observable.apply_layout(isa_circuit.layout) for observable in observables
>   ]
> 
>   job = estimator.run([(isa_circuit, mapped_observables)])
>   result = job.result()
> 
>   # This is the result of the entire submission.  You submitted one Pub,
>   # so this contains one inner result (and some metadata of its own).
> 
>   job_result = job.result()
> 
>   # This is the result from our single pub, which had five observables,
>   # so contains information on all five.
> 
>   pub_result = job.result()[0]
>   ```
### Hakbang 4. Suriin ang mga resulta
Ang hakbang ng pagsusuri ay karaniwang doon ka nagpo-post-process ng iyong mga resulta gamit, halimbawa, ang measurement error mitigation o zero noise extrapolation (ZNE). Maaari mong ipasok ang mga resultang ito sa isa pang workflow para sa karagdagang pagsusuri o maghanda ng graph ng mga pangunahing value at data. Sa pangkalahatan, ang hakbang na ito ay natatangi sa iyong problema. Para sa halimbawang ito, i-plot ang bawat isa sa mga expectation value na nasukat para sa ating Circuit.

Ang mga expectation value at standard deviation para sa mga observable na tinukoy mo sa Estimator ay ina-access sa pamamagitan ng mga attribute na `PubResult.data.evs` at `PubResult.data.stds` ng resulta ng job. Para makuha ang mga resulta mula sa Sampler, gamitin ang function na `PubResult.data.meas.get_counts()`, na magbabalik ng `dict` ng mga sukat sa anyo ng mga bitstring bilang mga key at mga count bilang kanilang kaukulang value. Para sa karagdagang impormasyon, tingnan ang [Magsimula sa Sampler.](/guides/get-started-with-primitives#get-started-with-sampler)

In [8]:
# Make sure the results follow the claim from the previous markdown cell.
# This can happen when the device occasionally behaves strangely. If this cell
# fails, you may just need to run the notebook again.

_results = {obs: val for obs, val in zip(observables_labels, values)}
for _label in ["IZ", "IX", "ZI", "XI"]:
    assert abs(_results[_label]) < 0.2
for _label in ["XX", "ZZ"]:
    assert _results[_label] > 0.8

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg)

Mapapansin na para sa mga qubit 0 at 1, ang mga independyenteng expectation value ng parehong X at Z ay 0, habang ang mga correlation (`XX` at `ZZ`) ay 1. Ito ay isang palatandaan ng quantum entanglement.

In [ ]:
def get_qc_for_n_qubit_GHZ_state(n: int) -> QuantumCircuit:
    """This function will create a qiskit.QuantumCircuit (qc) for an n-qubit GHZ state.

    Args:
        n (int): Number of qubits in the n-qubit GHZ state

    Returns:
        QuantumCircuit: Quantum circuit that generate the n-qubit GHZ state, assuming all qubits start in the 0 state
    """
    if isinstance(n, int) and n >= 2:
        qc = QuantumCircuit(n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
    else:
        raise Exception("n is not a valid input")
    return qc


# Create a new circuit with two qubits (first argument) and two classical
# bits (second argument)
n = 100
qc = get_qc_for_n_qubit_GHZ_state(n)

## I-scale sa malaking bilang ng qubits
Sa quantum computing, ang utility-scale na trabaho ay napakahalaga para sa pag-unlad ng larangan. Ang ganitong trabaho ay nangangailangan ng mga computasyon na gawin sa mas malaking scale; nagtatrabaho sa mga Circuit na maaaring gumamit ng mahigit 100 qubit at mahigit 1000 gate. Ipinapakita ng halimbawang ito kung paano mo magagawa ang utility-scale na trabaho sa IBM&reg; QPU sa pamamagitan ng paglikha at pagsusuri ng isang 100-qubit GHZ state. Gumagamit ito ng Qiskit patterns workflow at nagtatapos sa pagsukat ng expectation value na $\langle Z_0 Z_i \rangle$ para sa bawat qubit.

### Hakbang 1. I-map ang problema
Sumulat ng function na nagbabalik ng `QuantumCircuit` na naghahanda ng $n$-qubit GHZ state (mahalagang isang extended Bell state), tapos gamitin ang function na iyon para maghanda ng 100-qubit GHZ state at mangolekta ng mga observable na susukatín.

In [ ]:
# ZZII...II, ZIZI...II, ... , ZIII...IZ
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]
print(operator_strings)
print(len(operator_strings))

operators = [SparsePauliOp(operator) for operator in operator_strings]

['ZZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII

Susunod, i-map sa mga operator na interesado tayo. Ang halimbawang ito ay gumagamit ng mga `ZZ` operator sa pagitan ng mga qubit para suriin ang gawi habang lalong lumalayo ang mga ito sa isa't isa. Ang mga lalong hindi tumpak (sira) na expectation value sa pagitan ng mga malalayong qubit ay magpapakita ng antas ng noise na naroroon.

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(
    simulator=False, operational=True, min_num_qubits=100
)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

isa_circuit = pm.run(qc)
isa_operators_list = [op.apply_layout(isa_circuit.layout) for op in operators]

### Step 3. Execute on hardware

Submit the job and enable error suppression by using a technique to reduce errors called [dynamical decoupling.](../api/qiskit-ibm-runtime/options-dynamical-decoupling-options) The resilience level specifies how much resilience to build against errors. Higher levels generate more accurate results, at the expense of longer processing times.  For further explanation of the options set in the following code, see [Configure error mitigation for Qiskit Runtime.](/docs/guides/configure-error-mitigation)

In [ ]:
options = EstimatorOptions()
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"

# Create an Estimator object
estimator = Estimator(backend, options=options)

In [13]:
# Submit the circuit to Estimator
job = estimator.run([(isa_circuit, isa_operators_list)])
job_id = job.job_id()
print(job_id)

d5k9mmqvcahs73a1ni3g


### Hakbang 3. Mag-execute sa hardware
I-submit ang job at i-enable ang error suppression sa pamamagitan ng paggamit ng teknik para mabawasan ang mga error na tinatawag na [dynamical decoupling.](../api/qiskit-ibm-runtime/options-dynamical-decoupling-options) Tinutukoy ng resilience level kung gaano karaming resilience ang buuin laban sa mga error. Ang mas mataas na antas ay nagbibigay ng mas tumpak na mga resulta, ngunit nangangailangan ng mas mahabang oras ng pagproseso. Para sa karagdagang paliwanag ng mga opsyon na nakatakda sa sumusunod na code, tingnan ang [I-configure ang error mitigation para sa Qiskit Runtime.](/guides/configure-error-mitigation)

In [ ]:
# data
data = list(range(1, len(operators) + 1))  # Distance between the Z operators
result = job.result()[0]
values = result.data.evs  # Expectation value at each Z operator.
values = [
    v / values[0] for v in values
]  # Normalize the expectation values to evaluate how they decay with distance.

# plotting graph
plt.plot(data, values, marker="o", label="100-qubit GHZ state")
plt.xlabel("Distance between qubits $i$")
plt.ylabel(r"$\langle Z_i Z_0 \rangle / \langle Z_1 Z_0 \rangle $")
plt.legend()
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/de91ebd0-0.svg" alt="Output of the previous code cell" />

The previous plot shows that as the distance between qubits increases, the signal decays because of the presence of noise.

## Next steps

<Admonition type="tip" title="Recommendations">
  -   Try one of these tutorials:
      - [Ground-state energy estimation of the Heisenberg chain with VQE](/docs/tutorials/spin-chain-vqe)
      - Solve optimization problems using [QAOA](/docs/tutorials/quantum-approximate-optimization-algorithm)
      - Train [quantum kernel](/docs/tutorials/quantum-kernel-training) models for machine learning tasks
  - Find detailed installation instructions in the [Install Qiskit](/docs/guides/install-qiskit) guide.
  - If you prefer not to install Qiskit locally, read about options to use Qiskit in an [online development environment.](/docs/guides/online-lab-environments)
  - To save multiple account credentials or to specify other account options, see detailed instructions in the [Save your login credentials](/docs/guides/save-credentials#save-your-access-credentials) guide.
</Admonition>